# IHMM Demonstration Notebook

This notebook demonstrates how to use the IHMM implementation for fixation detection, including:
- Generating synthetic data
- Running the IHMM pipeline
- Visualizing results


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import pymovements as pm
# import your functions
from pymovements.events.detection.ihmm import compute_hmm, ihmm
from pymovements.gaze.experiment import Experiment

%config InlineBackend.figure_format = 'svg'

## Load Toy Dataset

In [ ]:
# Define the experimental setup
experiment = Experiment(
    screen_width_px=1280,
    screen_height_px=1024,
    screen_width_cm=38,
    screen_height_cm=30.2,
    distance_cm=68,
    origin="upper left",
    sampling_rate=250.0,
)

# Load gaze data from a CSV file and initialize three Gaze objects to test different modes.
gaze1 = pm.gaze.from_csv(
    "./gaze-toy-example.csv",
    experiment=experiment,
    time_column="time",
    pixel_columns=["x", "y"],
)

gaze2 = pm.gaze.from_csv(
    "./gaze-toy-example.csv",
    experiment=experiment,
    time_column="time",
    pixel_columns=["x", "y"],
)
gaze3 = pm.gaze.from_csv(
    "./gaze-toy-example.csv",
    experiment=experiment,
    time_column="time",
    pixel_columns=["x", "y"],
)

# Convert pixel coordinates to degrees of visual angle (dva).
# Requires a valid Experiment with screen geometry and distance.
gaze1.pix2deg()
gaze1.pos2vel()

gaze2.pix2deg()
gaze2.pos2vel()

gaze3.pix2deg()
gaze3.pos2vel()

gaze1

# The function parameters are as such
```
def ihmm(
        positions: list[list[float]] | list[tuple[float, float]] | np.ndarray,
        timesteps: list[int] | np.ndarray | None = None,
        mu: list[float] | np.ndarray | None = None,
        sigma: list[float] | np.ndarray | None = None,
        init_state: list[float] | np.ndarray | None = None,
        transition_probabilities: list[list[float]] | np.ndarray | None = None,
        reestimation_max_iters: int = 100,
        initialization: str | None = None,
        verbose: bool = False,
        name: str = 'fixation',
) -> Events:
```

## Run IHMM with default parameters

In [ ]:
dict = {'mu': [2.0140785987072225, 69.41529375180251], 'sigma': [1.3220152347857494, 87.32409626093246], 'init': [1.e+00, 1.e-12], 'trans': [[0.97360507, 0.02639493],
                                                                                                                                             [0.07593547, 0.92406453]]}

gaze1.detect("idt", name="idt_fixations")

# gaze1.events.frame.write_csv("outputIDT.csv")

# Run IHMM with reestimation

In [ ]:
# Use the "initialization" parameter to specify run the Baum-Welch
# algorithm to obtain optimal starting parameters

# Flag "verbose" as True to print and see the optimal obtained parameters

# The parameter "reestimation_max_iters" is used to stop the Baum-Welch
# algorithm if it runs too long,

# it should be need only for very large datasets as the algorithm usually
# reached convergence quickly
dict = {'mu': [2.0140785987072225, 69.41529375180251], 'sigma': [1.3220152347857494, 87.32409626093246], 'init': [
    0.9999999999992323, 1.000000000000001e-12], 'trans': [[0.80, 0.20], [0.20, 0.80]]}
gaze2.detect(
    "ihmm",
    reestimation=True,
    verbose=True,
    reestimation_max_iters=1000,
    name="fixation_ihmm")

# gaze2.events.frame.write_csv("output.csv")

In [ ]:
#

mu = [10, 300]

sigma = [11, 386]

init = [0.5, 0.5]

trans = [[0.95, 0.05], [0.05, 0.95]]


# gaze3.detect("ihmm", mu=mu, sigma=sigma, init_state = init, transition_probabilities = trans, name="fixation_ihmm")

# gaze3.events

## Visualize Detected events

In [ ]:
# visualize the data
pm.plotting.traceplot(gaze1)

In [ ]:
gaze1.compute_event_properties(("location", {"position_column": "pixel"}))

pm.plotting.scanpathplot(gaze1, event_name="idt_fixations")

In [ ]:
gaze2.compute_event_properties(("location", {"position_column": "pixel"}))

pm.plotting.scanpathplot(gaze2, event_name="fixation_ihmm")

In [ ]:
# gaze3.compute_event_properties(("location", {"position_column": "pixel"}))

# pm.plotting.scanpathplot(gaze3, event_name="fixation_ihmm")